In [1]:
#Bases de Dados 
from pathlib import Path
import pandas as pd

In [2]:
path = Path("C:/Users/silma/Projetos/vigimed/data/03_gold")
df_analytical = pd.read_parquet(path/'fat_analytical/fat_analytical.parquet')
df_analytical.head()

,ID,SEXO,ATC_LEVEL_4,ATC_LEVEL_5,REACAO_LLT,REACAO_PT
0,BR-ANVISA-300066865,Masculino,TUMOR NECROSIS FACTOR ALPHA (TNF-ALPHA) INHIBI...,L04AB04_Adalimumab,Lesão da medula espinhal na região lombar,Lesão da medula espinhal na região lombar
1,BR-ANVISA-300066865,Masculino,TUMOR NECROSIS FACTOR ALPHA (TNF-ALPHA) INHIBI...,L04AB04_Adalimumab,Erupção no local de injeção,Erupção cutânea no local de injeção
2,BR-ANVISA-300066865,Masculino,TUMOR NECROSIS FACTOR ALPHA (TNF-ALPHA) INHIBI...,L04AB04_Adalimumab,Trombo apical,Trombose cardíaca ventricular
3,BR-ANVISA-300067135,Feminino,TUMOR NECROSIS FACTOR ALPHA (TNF-ALPHA) INHIBI...,L04AB04_Adalimumab,Urticária no local da injeção,Urticária no local de injeção
4,BR-ANVISA-300067135,Feminino,TUMOR NECROSIS FACTOR ALPHA (TNF-ALPHA) INHIBI...,L04AB04_Adalimumab,Peptídeo natriurético cerebral aumentado,Peptídeo natriurético cerebral aumentado


In [3]:
df_analytical.ATC_LEVEL_5.value_counts()

ATC_LEVEL_5
Outros                        2306743
L04AB02_Infliximab              53915
L04AB06_Golimumab               12092
L04AB04_Adalimumab              11835
L01FA01_Rituximab                9912
L04AF03_Upadacitinib             7354
L04AB05_Certolizumab Pegol       5442
L04AB01_Etanercept               3700
L04AC07_Tocilizumab              3270
L04AF02_Baricitinib               738
L04AA24_Abatacept                 676
Name: count, dtype: int64

In [4]:
def v0(df, drug_col, event_col, drug_name, event_name):
    """
    PT: Cria uma tabela de contingência 2x2 para análise de desproporcionalidade.
    EN: Creates a 2x2 contingency table for disproportionality analysis.

    Parameters:
    df (pd.DataFrame): Database / Banco de dados.
    drug_col (str): Column with drug names / Coluna com nomes dos fármacos.
    event_col (str): Column with adverse events / Coluna com eventos adversos.
    drug_name (str): Target drug / Fármaco de interesse.
    event_name (str): Target event / Evento de interesse.
    """

    # a: Target drug AND Target event
    # a: Fármaco de interesse E Evento de interesse
    a = df[(df[drug_col] == drug_name) & (df[event_col] == event_name)].shape[0]

    # b: Target drug AND Other events
    # b: Fármaco de interesse E Outros eventos
    b = df[(df[drug_col] == drug_name) & (df[event_col] != event_name)].shape[0]

    # c: Other drugs AND Target event
    # c: Outros fármacos E Evento de interesse
    c = df[(df[drug_col] != drug_name) & (df[event_col] == event_name)].shape[0]

    # d: Other drugs AND Other events
    # d: Outros fármacos E Outros eventos
    d = df[(df[drug_col] != drug_name) & (df[event_col] != event_name)].shape[0]

    # PT: Retorna os valores organizados para os cálculos de ROR, PRR, IC, etc.
    # EN: Returns organized values for ROR, PRR, IC, etc., calculations.
    return pd.DataFrame({
        'drug': [drug_name],
        'event': [event_name],
        'a': [a], 'b': [b], 'c': [c], 'd': [d]
    })

In [5]:
def v1(df, drug_col, event_col, drug_name, event_name):
    """
    PT: Cria uma tabela de contingência 2x2 para análise de desproporcionalidade.
    EN: Creates a 2x2 contingency table for disproportionality analysis.

    Parameters:
    df (pd.DataFrame): Database / Banco de dados.
    drug_col (str): Column with drug names / Coluna com nomes dos fármacos.
    event_col (str): Column with adverse events / Coluna com eventos adversos.
    drug_name (str): Target drug / Fármaco de interesse.
    event_name (str): Target event / Evento de interesse.
    """

    # a: Target drug AND Target event
    # a: Fármaco de interesse E Evento de interesse
    a = df[(df[drug_col] == drug_name) & (df[event_col] == event_name)].ID.nunique()

    # b: Target drug AND Other events
    # b: Fármaco de interesse E Outros eventos
    b = df[(df[drug_col] == drug_name) & (df[event_col] != event_name)].ID.nunique()

    # c: Other drugs AND Target event
    # c: Outros fármacos E Evento de interesse
    c = df[(df[drug_col] != drug_name) & (df[event_col] == event_name)].ID.nunique()

    # d: Other drugs AND Other events
    # d: Outros fármacos E Outros eventos
    d = df[(df[drug_col] != drug_name) & (df[event_col] != event_name)].ID.nunique()

    # PT: Retorna os valores organizados para os cálculos de ROR, PRR, IC, etc.
    # EN: Returns organized values for ROR, PRR, IC, etc., calculations.
    return pd.DataFrame({
        'drug': [drug_name],
        'event': [event_name],
        'a': [a], 'b': [b], 'c': [c], 'd': [d]
    })

In [9]:
def v2(
    df: pd.DataFrame,
    case_col: str,
    drug_col: str,
    event_col: str,
    drug_name: str,
    event_name: str,
    dropna: bool = True,
) -> pd.DataFrame:
    """
    PT: Cria uma tabela 2x2 contando CASOS DISTINTOS (ID) para um par (drug,event).
    EN: Builds a 2x2 table counting DISTINCT cases (ID) for a (drug,event) pair.
    """

    use_cols = [case_col, drug_col, event_col]
    x = df.loc[:, use_cols].copy()

    if dropna:
        x = x.dropna(subset=use_cols)

    if x.empty:
        return pd.DataFrame(
            {"drug": [drug_name], "event": [event_name], "a": [0], "b": [0], "c": [0], "d": [0], "N_total": [0]}
        )

    # Total de casos distintos no dataset (ou no estrato)
    N_total = x[case_col].nunique()

    # IDs com droga alvo (independente do evento)
    ids_drug = x.loc[x[drug_col] == drug_name, case_col].drop_duplicates()
    n_drug = ids_drug.nunique()

    # IDs com evento alvo (independente da droga)
    ids_event = x.loc[x[event_col] == event_name, case_col].drop_duplicates()
    n_event = ids_event.nunique()

    # a: IDs com (droga alvo E evento alvo)
    # dedup por (ID, drug, event) evita inflar por repetição
    a = (
        x.loc[(x[drug_col] == drug_name) & (x[event_col] == event_name), [case_col, drug_col, event_col]]
        .drop_duplicates()
        .loc[:, case_col]
        .nunique()
    )

    # b,c,d derivados das marginais (não duplica)
    b = n_drug - a
    c = n_event - a
    d = N_total - a - b - c

    # Segurança: evita negativos por inconsistência/NA
    b = max(int(b), 0)
    c = max(int(c), 0)
    d = max(int(d), 0)

    return pd.DataFrame({
        "drug": [drug_name],
        "event": [event_name],
        "a": [int(a)],
        "b": [int(b)],
        "c": [int(c)],
        "d": [int(d)],
        "N_total": [int(N_total)],
        "n_drug": [int(n_drug)],
        "n_event": [int(n_event)],
    })


In [17]:
import pandas as pd

def v3(
    df: pd.DataFrame,
    case_col: str,
    drug_col: str,
    event_col: str,
    drug_name: str,
    event_name: str,
    dropna: bool = True,
) -> pd.DataFrame:
    """
    PT: 2x2 por par (drug,event) contando IDs em nível de caso (sem dupla contagem).
    EN: 2x2 for (drug,event) counting IDs at case level (no double counting).
    """

    x = df[[case_col, drug_col, event_col]].copy()
    if dropna:
        x = x.dropna(subset=[case_col, drug_col, event_col])

    if x.empty:
        return pd.DataFrame({"drug":[drug_name], "event":[event_name], "a":[0], "b":[0], "c":[0], "d":[0]})

    # Flags por linha
    x["_has_drug"] = x[drug_col].eq(drug_name)
    x["_has_event"] = x[event_col].eq(event_name)

    # Agrega para o nível do caso (ID)
    by_case = x.groupby(case_col, as_index=False)[["_has_drug", "_has_event"]].any()

    has_drug = by_case["_has_drug"]
    has_event = by_case["_has_event"]

    a = int((has_drug & has_event).sum())
    b = int((has_drug & ~has_event).sum())
    c = int((~has_drug & has_event).sum())
    d = int((~has_drug & ~has_event).sum())

    return pd.DataFrame({
        "drug": [drug_name],
        "event": [event_name],
        "a": [a], "b": [b], "c": [c], "d": [d],
        "N_total": [int(len(by_case))]
    })


In [6]:
df_v0 = v0(
    df = df_analytical, 
    drug_col = 'ATC_LEVEL_5',
    event_col = 'REACAO_PT',
    drug_name = 'L04AB04_Adalimumab',
    event_name = 'Úlcera intestinal'
)
df_v0

,drug,event,a,b,c,d
0,L04AB04_Adalimumab,Úlcera intestinal,19,11816,8368,2395474


In [8]:
df_v1 = v1(
    df = df_analytical, 
    drug_col = 'ATC_LEVEL_5',
    event_col = 'REACAO_PT',
    drug_name = 'L04AB04_Adalimumab',
    event_name = 'Úlcera intestinal'
)
df_v1

,drug,event,a,b,c,d
0,L04AB04_Adalimumab,Úlcera intestinal,13,1518,4450,285164


In [10]:
df_v2 = v2(
    df = df_analytical,
    case_col = 'ID',
    drug_col = 'ATC_LEVEL_5',
    event_col = 'REACAO_PT',
    drug_name = 'L04AB04_Adalimumab',
    event_name = 'Úlcera intestinal'
)
df_v2

,drug,event,a,b,c,d,N_total,n_drug,n_event
0,L04AB04_Adalimumab,Úlcera intestinal,13,1506,4449,280571,286539,1519,4462


In [18]:
df_v3 = v3(
    df = df_analytical,
    case_col = 'ID',
    drug_col = 'ATC_LEVEL_5',
    event_col = 'REACAO_PT',
    drug_name = 'L04AB04_Adalimumab',
    event_name = 'Úlcera intestinal'
)
df_v3

,drug,event,a,b,c,d,N_total
0,L04AB04_Adalimumab,Úlcera intestinal,13,1506,4449,280571,286539


## Validação SQL

In [12]:
import sys
import pandas as pd
 
sys.path.append('../../../src/')
 
# Configurações de exibição
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.displaylimit = 100
%config SqlMagic.named_parameters = "enabled"
%config SqlMagic.autocommit = True

# Banco em arquivo (persiste durante a sessão).
#%sql duckdb:///content/vigimed.duckdb
# Se preferir memória: 
%sql duckdb:///:memory:

Connecting to 'duckdb:///:memory:'

In [13]:
%%sql
DROP TABLE IF EXISTS fat_analytical;
CREATE TABLE fat_analytical AS
SELECT * FROM read_parquet("C:/Users/silma/Projetos/vigimed/data/03_gold/fat_analytical/fat_analytical.parquet");

Running query in 'duckdb:///:memory:'

,Success


In [ ]:
%%sql
select count() as A,count(distinct ID) as A_distinct from fat_analytical 
where ATC_LEVEL_5 = 'L04AB04_Adalimumab' and REACAO_PT = 'Úlcera intestinal'

union all

select count() as B,count(distinct ID) as b_distinct from fat_analytical 
where ATC_LEVEL_5 = 'L04AB04_Adalimumab' and REACAO_PT <> 'Úlcera intestinal'

union all

select count() as C,count(distinct ID) as C_distinct from fat_analytical 
where ATC_LEVEL_5 <> 'L04AB04_Adalimumab' and REACAO_PT = 'Úlcera intestinal'

union all

select count() as D,count(distinct ID) as D_distinct from fat_analytical 
where ATC_LEVEL_5 <> 'L04AB04_Adalimumab' and REACAO_PT <> 'Úlcera intestinal'

Running query in 'duckdb:///:memory:'

,A,A_distinct
0,19,13
1,11816,1518
2,8368,4450
3,2395474,285164
